# Capítulo 6 · K-means

¿Qué ocurre si representamos cada grupo con un solo punto? **K-means** resume cada grupo con un **centroide**, el promedio de sus observaciones en cada variable. Usaremos las dos lunas para ver qué formas produce esta regla y las flores de Iris para interpretar un centroide en centímetros.

Esta lectura se puede ejecutar por separado. Las etiquetas de origen de las lunas y los nombres de las especies servirán para interpretar el resultado después del agrupamiento.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.datasets import load_iris, make_moons
from sklearn.preprocessing import StandardScaler

## 1. Dos lunas · ¿puede un centro representar todo un arco?

Elegimos de antemano el número de grupos, **k**. Para entender un ajuste con dos grupos, imagina dos centros iniciales sobre la nube de puntos.

1. Cada punto se asigna al centro más cercano según la distancia euclidiana.
2. Cada centro se mueve al promedio de los puntos que recibió.
3. Se repiten ambas operaciones hasta que los centros se estabilizan o se alcanza el límite de iteraciones.

El objetivo es reducir la **inercia**, la suma de las distancias al cuadrado de cada punto al centro asignado. La posición inicial influye en el resultado. Con `n_init=10` probamos diez inicializaciones y conservamos la de menor inercia. La semilla permite repetir la demostración.

In [ ]:
X_lunas, luna_origen = make_moons(n_samples=200, noise=0.08, random_state=42)

In [ ]:
modelo_lunas = KMeans(n_clusters=2, n_init=10, random_state=42) 
grupos_lunas = modelo_lunas.fit_predict(X_lunas)
centros_lunas = modelo_lunas.cluster_centers_
print("Coordenadas de los dos centroides:")
print(centros_lunas.round(2))

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(9, 4), sharex=True, sharey=True)
for eje, titulo, colores in zip(ejes, ["Lunas de origen", "Grupos de K-means"],
                                [luna_origen, grupos_lunas]):
    eje.scatter(X_lunas[:, 0], X_lunas[:, 1], c=colores, cmap="viridis", s=20)
    eje.set(title=titulo, xlabel="Coordenada x", aspect="equal")
ejes[0].set_ylabel("Coordenada y")
ejes[1].scatter(centros_lunas[:, 0], centros_lunas[:, 1], marker="X", s=160,
                color="black", edgecolor="white", label="Centroides")
ejes[1].legend()
fig.tight_layout()
plt.show()

Sigue la luna superior de un extremo al otro. En el panel derecho cambia de color a mitad del recorrido. Imagina la recta perpendicular que cruza el punto medio entre las dos X. Sobre esa recta, ambos centros están a la misma distancia. A cada lado gana un centro distinto. Así se explica el corte que atraviesa los arcos aun cuando pedimos dos grupos.

## 2. Iris · ¿cómo es la flor promedio de cada grupo?

Agruparemos 150 flores con las cuatro medidas de sépalo y pétalo. La dispersión de estas medidas varía incluso estando todas en centímetros. Para que una variable muy dispersa tenga un peso comparable al de las demás, centramos cada columna y la dividimos por su desviación estándar.

Elegimos `k=3` para poder contrastar los grupos con las tres especies conocidas al final. El modelo recibe únicamente las cuatro columnas de medidas.

In [ ]:
iris = load_iris(as_frame=True)
medidas = iris.data.copy()
medidas.columns = ["longitud_sépalo", "ancho_sépalo", "longitud_pétalo", "ancho_pétalo"]
escalador = StandardScaler()
X_iris = escalador.fit_transform(medidas)
medidas.head()

In [ ]:
modelo_iris = KMeans(n_clusters=3, n_init=10, random_state=42)
grupos_iris = modelo_iris.fit_predict(X_iris)

# Devolvemos los centroides a centímetros para describir las flores.
centros_cm = pd.DataFrame(escalador.inverse_transform(modelo_iris.cluster_centers_),
                         columns=medidas.columns)
centros_cm.index.name = "Grupo"
centros_cm.round(2)

Busca la fila con la menor longitud promedio del pétalo. ¿También tiene el pétalo más estrecho? La siguiente figura sitúa esos promedios entre las flores. Usamos dos ejes para dibujar una asignación que se calculó en cuatro dimensiones.

In [ ]:
fig, eje = plt.subplots(figsize=(6, 4))
for grupo in centros_cm.index:
    flores = medidas[grupos_iris == grupo]
    eje.scatter(flores["longitud_pétalo"], flores["ancho_pétalo"], alpha=0.7, label=f"Grupo {grupo}")
eje.scatter(centros_cm["longitud_pétalo"], centros_cm["ancho_pétalo"],
            marker="X", s=160, color="black", edgecolor="white", label="Centroides")
eje.set(title="Flores y centroides vistos a través del pétalo",
        xlabel="Longitud del pétalo (cm)", ylabel="Ancho del pétalo (cm)")
eje.legend()
fig.tight_layout()
plt.show()

In [ ]:
especie = iris.target_names[iris.target]
pd.crosstab(especie, grupos_iris, rownames=["Especie"], colnames=["Grupo"])

La fila de *setosa* se concentra en el grupo de pétalos cortos. Las filas de *versicolor* y *virginica* comparten grupos. Para comparar este resultado con el de Ward, busca qué flores quedaron juntas en cada lectura. El número asignado a cada grupo es arbitrario.

## 3. ¿Qué cambia al aumentar k?

Volvamos a las mismas medidas estandarizadas de Iris. Si un grupo se divide en dos, cada parte puede tener un centro más cercano a sus puntos. Antes de ejecutar la celda, anticipa qué tendencia tendrá la inercia al pasar de uno a seis centros.

In [ ]:
valores_k = range(1, 7)
inercias = []
for k in valores_k:
    modelo = KMeans(n_clusters=k, n_init=10, random_state=42)
    modelo.fit(X_iris)
    inercias.append(modelo.inertia_)

fig, eje = plt.subplots(figsize=(6, 3.5))
eje.plot(valores_k, inercias, marker="o")
eje.set(title="Inercia en Iris", xlabel="Número de grupos (k)",
        ylabel="Inercia", xticks=list(valores_k))
fig.tight_layout()
plt.show()

En esta curva, el descenso más grande ocurre al pasar de uno a dos centros. Después, la mejora por cada centro adicional es menor. Buscar ese cambio de pendiente se conoce como **método del codo**. Puede haber varios cortes razonables. Para elegir uno, también importa si los perfiles de flores que obtienes responden a la pregunta del análisis.

**Para discutir en clase.** En las dos lunas, ¿qué pasaría si pusieras varios centroides a lo largo de cada arco? Explica cómo podría bajar la inercia aunque un mismo arco terminara repartido entre varios grupos.